In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    when,
    trim,
    lower,
    to_timestamp,
    min as spark_min,
    max as spark_max,
    sum as spark_sum,
    desc
)

SOURCE_FILE = "ecommerce_categorias.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false"
}

adls_options = get_adls_options()

print(f"Arquivo analisado: {SOURCE_FILE}")
print(f"Caminho Raw: {SOURCE_PATH}")

In [0]:
df_raw_categorias = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

print("Leitura da RAW concluída.")
print(f"Total de linhas: {df_raw_categorias.count()}")
print(f"Total de colunas: {len(df_raw_categorias.columns)}")

df_raw_categorias.printSchema()

display(df_raw_categorias.limit(20))

In [0]:
KEY_COLUMN = "id_categoria"

df_integridade_categorias = df_raw_categorias.select(
    count("*").alias("total_linhas"),
    countDistinct(KEY_COLUMN).alias("categorias_distintas"),
    (
        count("*") - countDistinct(KEY_COLUMN)
    ).alias("ids_categoria_duplicados"),
    spark_sum(
        when(col(KEY_COLUMN).isNull() | (trim(col(KEY_COLUMN)) == ""), 1).otherwise(0)
    ).alias("id_categoria_nulo_ou_vazio")
)

display(df_integridade_categorias)

In [0]:
df_categorias_pai = (
    df_raw_categorias
    .groupBy("id_categoria_pai")
    .agg(
        count("*").alias("qtd_categorias_filhas")
    )
    .orderBy(desc("qtd_categorias_filhas"))
)

display(df_categorias_pai)